# 03 — Generazione su HumanEval

Notebook per generare i completamenti di codice sui 164 problemi di **HumanEval**, usando ciascuno
dei 13 modelli (baseline fp16 + 12 quantizzati).


## 1. Setup

### Import e percorsi

In [1]:
import os
import json
import time
from datasets import load_dataset
from llama_cpp import Llama

MODELS_DIR = "../models"
RESULTS_DIR = "../results"
GENERATIONS_DIR = f"{RESULTS_DIR}/generations"

os.makedirs(GENERATIONS_DIR, exist_ok=True)

MODEL_F16 = f"{MODELS_DIR}/qwen2.5-coder-1.5b-f16.gguf"

CALIBRATIONS = ["random", "mixed", "code"]
QUANT_LEVELS = ["Q8_0", "Q4_K_M", "Q3_K_M", "Q2_K"]

# Lista completa dei 13 modelli da valutare: baseline + 12 quantizzati
MODEL_CONFIGS = [{"name": "baseline-f16", "path": MODEL_F16}]

for calib in CALIBRATIONS:
    for level in QUANT_LEVELS:
        name = f"{calib}-{level}"
        path = f"{MODELS_DIR}/qwen2.5-coder-1.5b-{calib}-{level}.gguf"
        MODEL_CONFIGS.append({"name": name, "path": path})

print(f"Modelli da valutare: {len(MODEL_CONFIGS)}")
for m in MODEL_CONFIGS:
    exists = "OK" if os.path.exists(m["path"]) else "MANCANTE"
    print(f"  [{exists}] {m['name']}")

Modelli da valutare: 13
  [OK] baseline-f16
  [OK] random-Q8_0
  [OK] random-Q4_K_M
  [OK] random-Q3_K_M
  [OK] random-Q2_K
  [OK] mixed-Q8_0
  [OK] mixed-Q4_K_M
  [OK] mixed-Q3_K_M
  [OK] mixed-Q2_K
  [OK] code-Q8_0
  [OK] code-Q4_K_M
  [OK] code-Q3_K_M
  [OK] code-Q2_K


## 2. Caricamento di HumanEval

HumanEval è composto da 164 problemi di programmazione, ciascuno con una funzione da completare
(firma + docstring) e dei test nascosti usati per valutare la correttezza.

In [3]:
humaneval = load_dataset("openai/openai_humaneval", split="test")
print(f"Numero di problemi HumanEval: {len(humaneval)}")

# Esempio di problema
example = humaneval[0]
print("\n=== Esempio (task_id:", example["task_id"], ") ===")
print(example["prompt"])

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Numero di problemi HumanEval: 164

=== Esempio (task_id: HumanEval/0 ) ===
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """



## 3. Costruzione dei prompt

Il campo `prompt` di HumanEval è già nel formato corretto (firma della funzione + docstring):
lo usiamo direttamente come input al modello, senza aggiungere altro. Il modello deve completare
il corpo della funzione in stile *completion*, non chat.

In [ ]:
def build_prompt(problem):
    return problem["prompt"]

# Sequenze di stop standard usate nella valutazione HumanEval:
# fermano la generazione non appena il modello inizia una nuova funzione/blocco che
# non fa più parte della soluzione.
STOP_SEQUENCES = [
    "\nclass ",
    "\ndef ",
    "\n#",
    "\nif __name__",
    "\nprint(",
    "\n```",
]

## 4. Funzione di generazione

Usiamo decoding **greedy** (`temperature=0`) per avere risultati riproducibili, come nel setup
standard di valutazione pass@1 con un solo campione per problema.

In [ ]:
MAX_NEW_TOKENS = 512

def load_model(model_path, n_ctx=2048):
    return Llama(
        model_path=model_path,
        n_ctx=n_ctx,
        n_gpu_layers=-1,   # scarica più layer possibile sulla GPU
        verbose=False,
    )

def generate_completion(llm, prompt):
    output = llm(
        prompt,
        max_tokens=MAX_NEW_TOKENS,
        temperature=0.0,
        stop=STOP_SEQUENCES,
        echo=False,
    )
    return output["choices"][0]["text"]

## 5. Test rapido della pipeline

Prima di lanciare la generazione completa (13 modelli × 164 problemi), verifichiamo che tutto
funzioni su un solo modello e pochi problemi.

In [6]:
TEST_N_PROBLEMS = 3

print("Caricamento modello di test (baseline fp16)...")
test_llm = load_model(MODEL_F16)

for i in range(TEST_N_PROBLEMS):
    problem = humaneval[i]
    prompt = build_prompt(problem)
    completion = generate_completion(test_llm, prompt)

    print(f"\n=== {problem['task_id']} ===")
    print("--- Prompt ---")
    print(prompt)
    print("--- Completamento generato ---")
    print(completion)

del test_llm  # libera la memoria prima della generazione completa

Caricamento modello di test (baseline fp16)...

=== HumanEval/0 ===
--- Prompt ---
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """

--- Completamento generato ---
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False



=== HumanEval/1 ===
--- Prompt ---
from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
    separate those group into separate strings and return the list of those.
    Separate groups are balanced (each open bra

## 6. Generazione completa

Per ciascuno dei 13 modelli: carichiamo il modello una volta, generiamo il completamento per
tutti i 164 problemi, salviamo il risultato in `results/generations/<nome_modello>.jsonl`.


In [7]:
generation_summary = []

for model_cfg in MODEL_CONFIGS:
    model_name = model_cfg["name"]
    model_path = model_cfg["path"]
    output_path = f"{GENERATIONS_DIR}/{model_name}.jsonl"

    if os.path.exists(output_path):
        print(f"=== {model_name}: già presente, salto ===")
        generation_summary.append({"model": model_name, "status": "skipped (già esistente)"})
        continue

    if not os.path.exists(model_path):
        print(f"=== {model_name}: file modello mancante, salto ===")
        generation_summary.append({"model": model_name, "status": "ERRORE: file mancante"})
        continue

    print(f"\n=== Generazione con: {model_name} ===")
    start_time = time.time()

    llm = load_model(model_path)

    with open(output_path, "w", encoding="utf-8") as f:
        for i, problem in enumerate(humaneval):
            prompt = build_prompt(problem)
            completion = generate_completion(llm, prompt)

            f.write(json.dumps({
                "task_id": problem["task_id"],
                "completion": completion,
            }) + "\n")

            if (i + 1) % 20 == 0:
                print(f"  {i + 1}/{len(humaneval)} problemi completati")

    del llm  # libera la memoria prima di caricare il modello successivo

    elapsed = time.time() - start_time
    print(f"Completato {model_name} in {elapsed / 60:.1f} minuti")
    generation_summary.append({"model": model_name, "status": f"OK ({elapsed / 60:.1f} min)"})

print("\nGenerazione completa per tutti i modelli.")


=== Generazione con: baseline-f16 ===
  20/164 problemi completati
  40/164 problemi completati
  60/164 problemi completati
  80/164 problemi completati
  100/164 problemi completati
  120/164 problemi completati
  140/164 problemi completati
  160/164 problemi completati
Completato baseline-f16 in 5.6 minuti

=== Generazione con: random-Q8_0 ===
  20/164 problemi completati
  40/164 problemi completati
  60/164 problemi completati
  80/164 problemi completati
  100/164 problemi completati
  120/164 problemi completati
  140/164 problemi completati
  160/164 problemi completati
Completato random-Q8_0 in 3.6 minuti

=== Generazione con: random-Q4_K_M ===
  20/164 problemi completati
  40/164 problemi completati
  60/164 problemi completati
  80/164 problemi completati
  100/164 problemi completati
  120/164 problemi completati
  140/164 problemi completati
  160/164 problemi completati
Completato random-Q4_K_M in 3.2 minuti

=== Generazione con: random-Q3_K_M ===
  20/164 problemi com

### Riepilogo della generazione

In [8]:
print(f"{'Modello':<18} Stato")
print("-" * 50)
for r in generation_summary:
    print(f"{r['model']:<18} {r['status']}")

Modello            Stato
--------------------------------------------------
baseline-f16       OK (5.6 min)
random-Q8_0        OK (3.6 min)
random-Q4_K_M      OK (3.2 min)
random-Q3_K_M      OK (3.7 min)
random-Q2_K        OK (5.0 min)
mixed-Q8_0         OK (3.9 min)
mixed-Q4_K_M       OK (3.4 min)
mixed-Q3_K_M       OK (4.9 min)
mixed-Q2_K         OK (3.9 min)
code-Q8_0          OK (3.8 min)
code-Q4_K_M        OK (3.0 min)
code-Q3_K_M        OK (3.8 min)
code-Q2_K          OK (3.8 min)


## 7. Controllo di sanità sui file generati

Verifichiamo che ogni file `.jsonl` contenga 164 righe (una per problema) e stampiamo un paio
di esempi per controllare a occhio che i completamenti abbiano senso.

In [9]:
for model_cfg in MODEL_CONFIGS:
    model_name = model_cfg["name"]
    output_path = f"{GENERATIONS_DIR}/{model_name}.jsonl"

    if not os.path.exists(output_path):
        print(f"{model_name}: file mancante")
        continue

    with open(output_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    status = "OK" if len(lines) == len(humaneval) else "ATTENZIONE numero righe inatteso"
    print(f"{model_name}: {len(lines)} righe [{status}]")

baseline-f16: 164 righe [OK]
random-Q8_0: 164 righe [OK]
random-Q4_K_M: 164 righe [OK]
random-Q3_K_M: 164 righe [OK]
random-Q2_K: 164 righe [OK]
mixed-Q8_0: 164 righe [OK]
mixed-Q4_K_M: 164 righe [OK]
mixed-Q3_K_M: 164 righe [OK]
mixed-Q2_K: 164 righe [OK]
code-Q8_0: 164 righe [OK]
code-Q4_K_M: 164 righe [OK]
code-Q3_K_M: 164 righe [OK]
code-Q2_K: 164 righe [OK]


In [10]:
# Esempio di completamento generato dalla baseline, per un controllo visivo
example_path = f"{GENERATIONS_DIR}/baseline-f16.jsonl"

if os.path.exists(example_path):
    with open(example_path, "r", encoding="utf-8") as f:
        first_line = json.loads(f.readline())

    print("=== Esempio di completamento (baseline-f16) ===")
    print("task_id:", first_line["task_id"])
    print("\n--- Completamento ---")
    print(first_line["completion"])
else:
    print("File della baseline non trovato: esegui prima la generazione completa (sezione 6).")

=== Esempio di completamento (baseline-f16) ===
task_id: HumanEval/0

--- Completamento ---
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False


